# Loading libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
#!pip install requests
import requests
import sys

# Loading datasets

#### **ClinPGx annontated dataset**




In [ ]:
## Load ClinPGx annotated dataset
clinpgx_protvar_data = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ProtVar/protvar_results_clinpgx.csv",
                       sep=","
                       )


In [ ]:
clinpgx_protvar_data = clinpgx_protvar_data[clinpgx_protvar_data['Notes'].isna()]

#### **Populational annontated dataset**



In [ ]:
## Load Populational annotated dataset
folder_path = "/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ProtVar"
csv_files = glob.glob(os.path.join(folder_path, 'protvar_results_1000genomes_*.csv'))

dfs = [pd.read_csv(file) for file in csv_files]
populational_protvar_data = pd.concat(dfs, ignore_index=True)

In [ ]:
populational_protvar_data = populational_protvar_data[populational_protvar_data['Notes'].isna()]

In [ ]:
# This popDataset_rsID_conversion dataset derives from the VCF files
popDataset_rsID_conversion = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/Populational_Data/PopDataset_rsID_conversion.txt",
                                         sep="\t",
                                         )

populational_protvar_data = populational_protvar_data.merge(popDataset_rsID_conversion, on='User_input', how='left')
populational_protvar_data['ID'] = populational_protvar_data['rsID']
populational_protvar_data = populational_protvar_data.drop(columns=['rsID'])

# Process individual datasets


In [ ]:
def rm_no_mapping(df):
    df = df[~(df['Notes'] == "No mapping found")]
    df['ID'] = df['ID'].replace('.', np.nan)
    return df


clinpgx_protvar_data = rm_no_mapping(clinpgx_protvar_data)
populational_protvar_data = rm_no_mapping(populational_protvar_data)

In [ ]:
# Adding rsIDs to where it was missing
# Regarding the Populational dataset, the VCF files did not have rsIDs for all the variants
def missing_rs_ids(df):
  list_df = df.loc[
      df['ID'].isna(),
      'User_input'
      ].to_list()
  return list_df

missing_rs_ids_clinpgx = missing_rs_ids(clinpgx_protvar_data)
missing_rs_ids_populational = missing_rs_ids(populational_protvar_data)

missing_rs_ids_list = list(set(missing_rs_ids_clinpgx + missing_rs_ids_populational))

with open('/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ProtVar/missing_rs_ids_protvar_dataset.txt', 'w') as f:
    for item in missing_rs_ids_list:
        f.write(f"{item}\n")

In [ ]:
# Some rsIDs were missing. I fetch this information using the Ensembl API (via the ensembl_api.py)
rs_ids_mapping_protvar_dataset = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ProtVar/rs_ids_mapping_protvar_dataset.txt",
                                             sep="\t",
                                             header=None,
                                             names = ['User_input', 'rsID'])

In [ ]:
mapping_dict = dict(zip(rs_ids_mapping_protvar_dataset['rsID'], rs_ids_mapping_protvar_dataset['User_input']))

def mapping_rsids(df):
    df['ID'] = df['User_input'].map(mapping_dict).fillna(df['ID'])
    return df

clinpgx_protvar_data = mapping_rsids(clinpgx_protvar_data)
populational_protvar_data = mapping_rsids(populational_protvar_data)

clinpgx_protvar_data['ID'].isna().sum()
populational_protvar_data['ID'].isna().sum()


In [ ]:
clinpgx_ids = clinpgx_protvar_data['User_input'].unique().tolist()
print(f"Number of unique variants in ClinPGx Protvar anotated dataset: {len(clinpgx_ids)}")

populational_ids = populational_protvar_data['User_input'].unique().tolist()
print(f"Number of unique variants in Populatonal Protvar anotated dataset: {len(populational_ids)}")

missing_ids = [x for x in clinpgx_ids if x not in populational_ids]
print(f"Number of unique variants present in ClinPGx but not in Populational: {len(missing_ids)}")

#### **Add missing functional information**


After annotating the ClinPGx and Populational datasets via the ProtVar API, I noticed that important functional features were not being retrieved (due to a bug in the Protvar UI). Therefore, I extracted this information using another feature of the ProtVar API (`protvar_functional_anno_api.py`)

In [ ]:
protvar_dataset_functional_annotated = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ProtVar/merged_clinpgx_protvar_data_functional_annotated.txt",
                                                     sep=";",
                                                     header=None,
                                                     names = ['User_input', 'Conservation_score', 'ESM1b_score', 'EVE_score(class)', 'popEVE_score', 'foldxDdg', 'foldxplddt']
                                                     )




In [ ]:
# ClinPGx dataset
clinpgx_protvar_data['User_input_decoy'] = clinpgx_protvar_data['Chromosome'].astype(str) + ":" + \
                                            clinpgx_protvar_data['Coordinate'].astype(str) + ":" + \
                                            clinpgx_protvar_data['Reference_allele'].astype(str) + ":" + \
                                            clinpgx_protvar_data['Alternative_allele'].astype(str)

clinpgx_protvar_data = clinpgx_protvar_data.merge(protvar_dataset_functional_annotated,
                                                  left_on='User_input_decoy',
                                                  right_on='User_input',
                                                  how='left')

clinpgx_protvar_data = clinpgx_protvar_data.drop(columns=['User_input_decoy', 'Conservation_score_x', 'EVE_score(class)_x',
                                                          'ESM1b_score_x', 'User_input_y', 'Foldx_prediction(foldxDdg;plddt)'])

clinpgx_protvar_data.columns = clinpgx_protvar_data.columns.str.replace('_[xy]', '', regex=True)



# Populational dataset
populational_protvar_data = populational_protvar_data.merge(protvar_dataset_functional_annotated,
                                                            on='User_input',
                                                            how='left')

populational_protvar_data = populational_protvar_data.drop(columns=['Conservation_score_x', 'EVE_score(class)_x','ESM1b_score_x', 'Foldx_prediction(foldxDdg;plddt)'])
populational_protvar_data.columns = populational_protvar_data.columns.str.replace('_[xy]', '', regex=True)



## Reordering columns
def reorder_cols(df):
    col_ddg = df.pop('foldxDdg')
    col_plddt = df.pop('foldxplddt')
    col_conservation = df.pop('Conservation_score')
    col_esm1b = df.pop('ESM1b_score')
    col_eve = df.pop('EVE_score(class)')
    col_popEVE = df.pop('popEVE_score')

    df.insert(32, 'foldxDdg', col_ddg)
    df.insert(33, 'foldxplddt', col_plddt)
    df.insert(34, 'Conservation_score', col_conservation)
    df.insert(36, 'ESM1b_score', col_esm1b)
    df.insert(37, 'EVE_score(class)', col_eve)
    df.insert(38, 'popEVE_score', col_popEVE)
    return df

clinpgx_protvar_data = reorder_cols(clinpgx_protvar_data)
populational_protvar_data = reorder_cols(populational_protvar_data)


After inspection of the datasets, and before inducing bias in pocket and protein interactions feature engineering, some variants were not annotated by the ProtVar (those whose feature 'Entry_last_updated' was NaN). So I excluded them from the datasets

In [ ]:
print(
    (clinpgx_protvar_data.isna().mean() * 100)
    .sort_values(ascending=False)
    .to_string()
)

In [ ]:
print(
    (populational_protvar_data.isna().mean() * 100)
    .sort_values(ascending=False)
    .to_string()
)

In [ ]:
clinpgx_protvar_data = clinpgx_protvar_data[clinpgx_protvar_data['Entry_last_updated'].notna()]
populational_protvar_data = populational_protvar_data[populational_protvar_data['Entry_last_updated'].notna()]

#### **Feature engineering: pockets and PPI**

In [ ]:
# Feature engineering: Pockets

def parse_global_pockets(pockets_str):
    if pd.isna(pockets_str) or pockets_str == '':
        return pd.Series({
            'num_pockets': 0,
            'top_pocket_score': 0,
            'mean_pocket_score': 0,
            'min_energy_per_vol': 0,
            'mean_energy_per_vol': 0,
            'max_rad_gyration': 0,
            'max_buriedness': 0,
            'mean_plddt_pockets': 0
        })

    pockets = pockets_str.split('|')

    scores, energies, rads, buriedness, plddts = [], [], [], [], []

    for p in pockets:
        p_dict = dict(item.split(':') for item in p.split(';'))

        scores.append(float(p_dict.get('score', 0)))
        energies.append(float(p_dict.get('energy_per_vol', 0)))
        rads.append(float(p_dict.get('rad_gyration', 0)))
        buriedness.append(float(p_dict.get('buriedness', 0)))
        plddts.append(float(p_dict.get('mean_plddt', 0)))

    return pd.Series({
        'num_pockets': len(pockets),
        'top_pocket_score': max(scores) if scores else 0,
        'mean_pocket_score': np.mean(scores) if scores else 0,
        'min_energy_per_vol': min(energies) if energies else 0,
        'mean_energy_per_vol': np.mean(energies) if energies else 0,
        'max_rad_gyration': max(rads) if rads else 0,
        'max_buriedness': max(buriedness) if buriedness else 0,
        'mean_plddt_pockets': np.mean(plddts) if plddts else 0
    })

def process_and_reorder(df, col_name='Predicted_pockets(energy;per_vol;score;resids)', target_index=30):
    new_features = df[col_name].apply(parse_global_pockets)

    df_cleaned = df.drop(columns=[col_name])

    new_feature_cols = new_features.columns.tolist()
    original_cols = df_cleaned.columns.tolist()

    new_column_order = original_cols[:target_index] + new_feature_cols + original_cols[target_index:]

    df_final = pd.concat([df_cleaned, new_features], axis=1)[new_column_order]

    return df_final


clinpgx_protvar_data_pocket = process_and_reorder(clinpgx_protvar_data, target_index=30)
populational_protvar_data_pocket = process_and_reorder(populational_protvar_data, target_index=30)


In [ ]:
# Feature engineering: PPI
# Protein-Protein known interactions
def parse_ppi(ppi_str):
    if pd.isna(ppi_str) or str(ppi_str).strip() == '':
        return pd.Series({
            'pp_known_num_interactions': 0,
            'pp_known_has_interactions': 0
        })

    interactions = str(ppi_str).split(';')

    return pd.Series({
        'pp_known_num_interactions': len(interactions),
        'pp_known_has_interactions': 1
    })

def process_ppi_column(df, col_name='Protein_interactions_PROTEIN(gene)', target_index=1):
    new_features = df[col_name].apply(parse_ppi)

    df_cleaned = df.drop(columns=[col_name])

    original_cols = df_cleaned.columns.tolist()
    new_feature_cols = new_features.columns.tolist()

    new_column_order = original_cols[:target_index] + new_feature_cols + original_cols[target_index:]

    df_final = pd.concat([df_cleaned, new_features], axis=1)[new_column_order]

    return df_final


clinpgx_protvar_data_pocket_ppi = process_ppi_column(clinpgx_protvar_data_pocket, target_index=29)
populational_protvar_data_pocket_ppi = process_ppi_column(populational_protvar_data_pocket, target_index=29)


# Protein-protein predicted interactions at the aminoacid level
def parse_global_predicted_interactions(interact_str):
    if pd.isna(interact_str) or str(interact_str).strip() == '':
        return pd.Series({
            'num_predicted_interactions': 0,
            'max_pDockQ': 0,
            'mean_pDockQ': 0
        })

    interactions = str(interact_str).split('|')
    pdockq_scores = []

    for interaction in interactions:
        parts = interaction.split(';')
        for part in parts:
            part = part.strip()
            if part.startswith('pDockQ:'):
                try:
                    pdockq_scores.append(float(part.split(':')[1]))
                except ValueError:
                    pass

    return pd.Series({
        'num_predicted_interactions': len(interactions),
        'max_pDockQ': max(pdockq_scores) if pdockq_scores else 0,
        'mean_pDockQ': np.mean(pdockq_scores) if pdockq_scores else 0
    })

def process_predicted_interactions(df, col_name='Predicted_interactions(chainA-chainB;a_resids;b_resids;pDockQ)', target_index=2):
    new_features = df[col_name].apply(parse_global_predicted_interactions)

    df_cleaned = df.drop(columns=[col_name])

    original_cols = df_cleaned.columns.tolist()
    new_feature_cols = new_features.columns.tolist()

    new_column_order = original_cols[:target_index] + new_feature_cols + original_cols[target_index:]

    df_final = pd.concat([df_cleaned, new_features], axis=1)[new_column_order]

    return df_final

clinpgx_protvar_data_pocket_ppi = process_predicted_interactions(clinpgx_protvar_data_pocket_ppi, target_index=39)
populational_protvar_data_pocket_ppi = process_predicted_interactions(populational_protvar_data_pocket_ppi, target_index=39)


In [ ]:
final_protvar_data = pd.concat([clinpgx_protvar_data_pocket_ppi, populational_protvar_data_pocket_ppi], ignore_index=True).drop_duplicates(subset=['User_input'])

final_protvar_data['AlphaMissense_pathogenicity(class)'] = final_protvar_data['AlphaMissense_pathogenicity(class)'].str.split('(').str[0]
final_protvar_data['AlphaMissense_pathogenicity(class)'] = pd.to_numeric(final_protvar_data['AlphaMissense_pathogenicity(class)'], errors='coerce')

final_protvar_data['User_input'] = final_protvar_data['User_input'] + ":" + final_protvar_data['Gene']



final_protvar_data.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/Datasets/ProtVar/final_protvar_data.csv",
                          index=False,
                          sep="\t",
                          na_rep="NA"
                          )